# Phase 1 Final Influence Package

This notebook is orchestration only. It runs the repository CLI for the final influence package after final comparator reconciliation has completed.

Scientific integrity rules:

- This notebook does not train or retrain any comparator.
- This notebook does not edit logits, fabricate leave-one-subject-out checks, or open headline claims.
- The runner computes subject-level and leave-one-subject-out diagnostics only from reconciled final comparator logits.
- If a single-subject influence ceiling is exceeded, the correct result is a blocked final influence manifest.
- Passing influence alone is not enough for a Phase 1 claim; controls, calibration, reporting, and final governance must still pass.


In [ ]:
# Cell 1 - Bootstrap repo and run unit tests.

from pathlib import Path
import base64
import getpass
import json
import os
import subprocess
import sys

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    pass

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752')

def run(cmd, cwd=None, check=True):
    display = []
    for item in cmd:
        text = str(item)
        if 'Authorization: Basic' in text:
            display.append('http.extraHeader=<redacted>')
        else:
            display.append(text)
    print('$ ' + ' '.join(display))
    result = subprocess.run(cmd, cwd=str(cwd) if cwd else None, text=True, check=check)
    return result

if not REPO_DIR.exists():
    token = getpass.getpass('Nhap GitHub token cho repo private: ')
    header = 'Authorization: Basic ' + base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    run(['git', '-c', f'http.extraHeader={header}', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)

run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)
unit_result = subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=str(REPO_DIR), text=True)
if unit_result.returncode != 0:
    raise subprocess.CalledProcessError(unit_result.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])


In [ ]:
# Cell 2 - Select reviewed source artifacts and keep launch disabled by default.

from pathlib import Path
import hashlib
import json

DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'

EXPECTED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PREREG_BUNDLE = ARTIFACT_ROOT / 'prereg/20260418T161442014597Z/prereg_bundle.json'

# Use None to resolve latest.txt, or pin an already reviewed comparator reconciliation run.
COMPARATOR_RECONCILIATION_RUN = Path('/content/drive/MyDrive/eeg-ds004752/artifacts/phase1_final_comparator_reconciliation/20260422T014337472987Z')

OUTPUT_ROOT = ARTIFACT_ROOT / 'phase1_final_influence'
PLAN_ROOT = ARTIFACT_ROOT / 'phase1_final_influence_plan'

RUN_FINAL_INFLUENCE = False
REQUIRED_ACK = 'I reviewed final influence and understand it computes leave-one-subject-out diagnostics only from final logits without opening claims'
MANUAL_LAUNCH_ACK = ''

FIX_MARKER = 'phase1_final_influence_v1_20260422'
print('Notebook fix marker:', FIX_MARKER)


In [ ]:
# Cell 3 - Resolve paths and validate prereg identity plus comparator reconciliation boundary.

def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def resolve_latest(root: Path) -> Path:
    latest = root / 'latest.txt'
    if latest.exists():
        return Path(latest.read_text(encoding='utf-8').strip())
    runs = sorted([p for p in root.iterdir() if p.is_dir()]) if root.exists() else []
    if not runs:
        raise FileNotFoundError(f'No runs found under {root}')
    return runs[-1]

PREREG_BUNDLE = Path(PREREG_BUNDLE)
COMPARATOR_RECONCILIATION_RUN = Path(COMPARATOR_RECONCILIATION_RUN) if COMPARATOR_RECONCILIATION_RUN else resolve_latest(ARTIFACT_ROOT / 'phase1_final_comparator_reconciliation')

bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_sha256 = sha256(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_PREREG_IDENTITY_HASH, 'Locked prereg identity hash mismatch.'

comp_summary = read_json(COMPARATOR_RECONCILIATION_RUN / 'phase1_final_comparator_reconciliation_summary.json')
comp_claim_state = read_json(COMPARATOR_RECONCILIATION_RUN / 'phase1_final_comparator_reconciled_claim_state.json')
assert comp_summary.get('status') == 'phase1_final_comparator_reconciliation_complete_claim_closed', comp_summary.get('status')
assert comp_summary.get('all_final_comparator_outputs_present') is True
assert comp_summary.get('runtime_comparator_logs_audited_for_all_required_comparators') is True
assert comp_summary.get('claim_ready') is False
assert comp_claim_state.get('claim_ready') is False
print('Comparator reconciliation run:', COMPARATOR_RECONCILIATION_RUN)


In [ ]:
# Cell 4 - Preflight runner/config availability and write a reviewable plan artifact.

from datetime import datetime, timezone

sys.path.insert(0, str(REPO_DIR))
preflight_blockers = []
try:
    from src.phase1.final_influence import run_phase1_final_influence  # noqa: F401
    runner_available = True
except Exception as exc:
    runner_available = False
    preflight_blockers.append(f'final influence runner import failed: {exc}')

required_repo_files = [
    REPO_DIR / 'src/phase1/final_influence.py',
    REPO_DIR / 'configs/phase1/final_influence.json',
    REPO_DIR / 'src/phase1/influence.py',
]
for path in required_repo_files:
    if not path.exists():
        preflight_blockers.append(f'missing repo file: {path.relative_to(REPO_DIR)}')

PLAN_ROOT.mkdir(parents=True, exist_ok=True)
plan_dir = PLAN_ROOT / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
plan_dir.mkdir(parents=True, exist_ok=False)

cmd = [
    'python', '-m', 'src.cli', 'phase1_final_influence',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--comparator-reconciliation-run', str(COMPARATOR_RECONCILIATION_RUN),
    '--output-root', str(OUTPUT_ROOT),
    '--influence-config', 'configs/phase1/final_influence.json',
    '--gate1-config', 'configs/gate1/decision_simulation.json',
    '--gate2-config', 'configs/gate2/synthetic_validation.json',
]

plan = {
    'status': 'phase1_final_influence_plan_recorded',
    'created_utc': plan_dir.name,
    'fix_marker': FIX_MARKER,
    'runner_available': runner_available,
    'run_flag': RUN_FINAL_INFLUENCE,
    'required_ack': REQUIRED_ACK,
    'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
    'preflight_blockers': preflight_blockers,
    'prereg_bundle': str(PREREG_BUNDLE),
    'locked_prereg_identity_hash': actual_prereg_hash,
    'raw_prereg_file_sha256': raw_prereg_sha256,
    'comparator_reconciliation_run': str(COMPARATOR_RECONCILIATION_RUN),
    'output_root': str(OUTPUT_ROOT),
    'command': cmd,
    'scientific_integrity_rule': 'Plan only. No influence result exists until the CLI command runs and writes artifacts.',
}
(plan_dir / 'phase1_final_influence_plan.json').write_text(json.dumps(plan, indent=2) + '
', encoding='utf-8')
print(json.dumps({k: plan[k] for k in ['status', 'runner_available', 'run_flag', 'ack_matched', 'preflight_blockers']}, indent=2))
if preflight_blockers:
    raise RuntimeError(preflight_blockers)


In [ ]:
# Cell 5 - Manual hold or launch the CLI-backed final influence runner.

if not RUN_FINAL_INFLUENCE:
    hold = {
        'status': 'phase1_final_influence_manual_hold',
        'run_flag': RUN_FINAL_INFLUENCE,
        'required_ack': REQUIRED_ACK,
        'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
        'plan_dir': str(plan_dir),
    }
    print(json.dumps(hold, indent=2))
elif MANUAL_LAUNCH_ACK != REQUIRED_ACK:
    raise RuntimeError('Manual launch acknowledgement mismatch. Do not launch final influence without explicit non-claim acknowledgement.')
else:
    launch_review = {
        'status': 'phase1_final_influence_launch_review_passed',
        'run_flag': RUN_FINAL_INFLUENCE,
        'ack_matched': True,
        'claim_ready_before_run': False,
    }
    (plan_dir / 'phase1_final_influence_launch_review.json').write_text(json.dumps(launch_review, indent=2) + '
', encoding='utf-8')
    run(cmd, cwd=REPO_DIR)
    print('Final influence command completed. Review generated artifacts before downstream governance reconciliation.')


In [ ]:
# Cell 6 - Inspect latest final influence output if execution was launched.

summary = None
latest_run = None
if RUN_FINAL_INFLUENCE:
    latest_run = resolve_latest(OUTPUT_ROOT)
    print('Latest final influence output:', latest_run)
    required_files = [
        'phase1_final_influence_inputs.json',
        'phase1_final_influence_summary.json',
        'phase1_final_influence_report.md',
        'phase1_final_influence_source_links.json',
        'phase1_final_influence_input_validation.json',
        'subject_level_fold_metrics.json',
        'leave_one_subject_out_deltas.json',
        'max_single_subject_contribution_share.json',
        'claim_state_leave_one_subject_out.json',
        'influence_veto_decision.json',
        'final_influence_manifest.json',
        'phase1_final_influence_claim_state.json',
    ]
    for name in required_files:
        print(name, 'exists =', (latest_run / name).exists())
    summary = read_json(latest_run / 'phase1_final_influence_summary.json')
    print(json.dumps({
        'status': summary.get('status'),
        'influence_package_passed': summary.get('influence_package_passed'),
        'leave_one_subject_out_executed': summary.get('leave_one_subject_out_executed'),
        'influence_ceiling': summary.get('influence_ceiling'),
        'max_single_subject_contribution_share': summary.get('max_single_subject_contribution_share'),
        'influence_vetoed': summary.get('influence_vetoed'),
        'claim_blockers': summary.get('claim_blockers'),
    }, indent=2))
else:
    print('Manual hold only; no final influence output inspected.')


In [ ]:
# Cell 7 - Assertions and non-claim review note.

if RUN_FINAL_INFLUENCE:
    assert summary is not None
    assert summary.get('claim_ready') is False
    assert summary.get('headline_phase1_claim_open') is False
    assert summary.get('full_phase1_claim_bearing_run_allowed') is False
    assert summary.get('leave_one_subject_out_executed') is True
    assert summary.get('status') in ['phase1_final_influence_complete_claim_closed', 'phase1_final_influence_blocked']
    manifest = read_json(latest_run / 'final_influence_manifest.json')
    assert manifest.get('claim_ready') is False
    assert manifest.get('smoke_artifacts_promoted') is False
    if summary.get('influence_package_passed') is False:
        assert summary.get('claim_blockers'), 'Blocked final influence must name blockers.'
    review_note = {
        'status': 'phase1_final_influence_review_recorded',
        'reviewed_run': str(latest_run),
        'scope': 'final subject-level and leave-one-subject-out influence diagnostics from reconciled logits',
        'influence_package_passed': summary.get('influence_package_passed'),
        'metrics_interpretation': {
            'allowed': 'Influence concentration diagnostics only.',
            'not_allowed': 'Do not use influence pass/fail as decoder efficacy, privileged-transfer efficacy, or full Phase 1 evidence.',
        },
        'next_allowed_steps': [
            'Rerun final governance reconciliation with final_influence_manifest if review is acceptable.',
            'Do not open headline claims while controls/calibration/reporting blockers remain.',
        ],
        'not_ok_to_claim': [
            'decoder efficacy',
            'A2d efficacy',
            'A3/A4 efficacy',
            'A4 superiority',
            'privileged-transfer efficacy',
            'full Phase 1 neural comparator performance',
        ],
    }
    (latest_run / 'phase1_final_influence_review_note.json').write_text(json.dumps(review_note, indent=2) + '
', encoding='utf-8')
    print('Review note written:', latest_run / 'phase1_final_influence_review_note.json')
    print(json.dumps(review_note, indent=2))
else:
    print('No assertions beyond plan/manual hold because RUN_FINAL_INFLUENCE is False.')


In [ ]:
# Cell 8 - Closeout.

print('================ Phase 1 Final Influence Closeout ================')
print('Notebook fix marker:', FIX_MARKER)
print('Plan source of truth:', plan_dir)
print('Runner available:', runner_available)
print('Run flag:', RUN_FINAL_INFLUENCE)
print('Expected locked prereg identity hash:', EXPECTED_PREREG_IDENTITY_HASH)
print('Locked prereg hash from bundle:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
print('Comparator reconciliation run:', COMPARATOR_RECONCILIATION_RUN)
print('Preflight blockers:', preflight_blockers)
if RUN_FINAL_INFLUENCE:
    print('Latest final influence output:', latest_run)
    print('Influence package passed:', summary.get('influence_package_passed'))
    print('Max single-subject contribution share:', summary.get('max_single_subject_contribution_share'))
    print('Influence vetoed:', summary.get('influence_vetoed'))
    print('Claim blockers:', summary.get('claim_blockers'))
    print('CHECK REQUIRED: Review final_influence_manifest.json and influence veto status before rerunning final governance reconciliation with --final-influence-manifest.')
else:
    print('HELD: Runner appears available, but influence was not launched because manual flag is False.')
    print('NEXT: review the plan, then rerun only with explicit non-claim acknowledgement if appropriate.')
print('NOT OK TO CLAIM: This notebook does not prove decoder efficacy, A2d efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')
